# CarVal Vision: YOLOv8 damage detection on Colab GPU
**Member 1 - Vision/Damage Detection.** Runs the same pipeline as `services/vision/training/train.py` on a Colab GPU.

**Before you start:**
1. Zip the YOLO dataset and upload it to Google Drive as `MyDrive/CarVal/CarDD_yolo.zip`:
   ```bash
   cd ~/Documents/BTP/datasets
   zip -rq CarDD_yolo.zip CarDD_yolo -x "*.cache" "CarDD_yolo/train_rfs.txt" "CarDD_yolo/data_rfs.yaml"
   ```
2. Get the code here in one of two ways:
   - **A (after the `nishitha` branch is pushed):** the clone cell below does it.
   - **B (before pushing):** zip the vision service and upload it as `MyDrive/CarVal/vision_code.zip`:
     ```bash
     cd ~/Documents/BTP/CarVal
     zip -rq vision_code.zip services/vision -x "services/vision/.venv/*" "services/vision/runs/*" "services/vision/weights/*" "*/__pycache__/*"
     ```

**Runtime → Change runtime type → T4 GPU** (or better), then run the cells top to bottom.
Runs are written straight to Drive (`MyDrive/CarVal/runs`), so nothing is lost if Colab disconnects.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/CarVal'
import os; os.makedirs(f'{DRIVE}/runs', exist_ok=True)

## 1. Code

In [ ]:
import os
CODE_SOURCE = 'drive'   # 'git' once the nishitha branch is pushed, else 'drive' (vision_code.zip)

if CODE_SOURCE == 'git':
    !rm -rf /content/CarVal && git clone -q -b nishitha https://github.com/shivam2718/CarVal.git /content/CarVal
else:
    !rm -rf /content/CarVal && mkdir -p /content/CarVal && unzip -q {DRIVE}/vision_code.zip -d /content/CarVal

VISION = '/content/CarVal/services/vision'
# Training output goes straight to Drive
!rm -rf {VISION}/runs && ln -s {DRIVE}/runs {VISION}/runs
%cd {VISION}
!ls

In [ ]:
!pip install -q "ultralytics>=8.4.0,<9.0"
import ultralytics; ultralytics.checks()

## 2. Dataset

In [ ]:
!cp {DRIVE}/CarDD_yolo.zip /content/ && unzip -q -o /content/CarDD_yolo.zip -d /content/ && rm /content/CarDD_yolo.zip
# data.yaml still points at the Mac path; repoint it
import yaml
p = '/content/CarDD_yolo/data.yaml'
d = yaml.safe_load(open(p)); d['path'] = '/content/CarDD_yolo'
yaml.safe_dump(d, open(p, 'w'), sort_keys=False)
print(open(p).read())
!for s in train val test; do echo "$s: $(ls /content/CarDD_yolo/images/$s | wc -l) images"; done
DATA = '/content/CarDD_yolo/data.yaml'

## 3. Baseline
COCO-pretrained yolov8n, default hyperparameters, no class balancing, 30 epochs (about 30 min on a T4).
Saved as `weights/baseline_yolov8n.pt`; it does not replace `weights/best.pt`.

In [ ]:
# Defaults so this cell also works after a Colab restart
DRIVE = '/content/drive/MyDrive/CarVal'
VISION = '/content/CarVal/services/vision'
DATA = '/content/CarDD_yolo/data.yaml'
%cd {VISION}
!python training/train.py --preset baseline --data {DATA} --device auto --batch 32 --workers 8

## 4. Fine-tune
Repeat-factor sampling for rare classes (tire, glass, lamp, crack), cosine LR, early stopping (patience 20).
yolov8s gives better accuracy than n and is still fast for the API (about 2 h on a T4 for 80 epochs).
Use `--model yolov8n.pt` if you want the smaller model.

In [ ]:
# Defaults so this cell also works after a Colab restart
DRIVE = '/content/drive/MyDrive/CarVal'
VISION = '/content/CarVal/services/vision'
DATA = '/content/CarDD_yolo/data.yaml'
%cd {VISION}
!python training/train.py --preset finetune --model yolov8s.pt --data {DATA} --device auto --batch 32 --workers 8

**If Colab disconnected mid-training**, reconnect, run sections 1-2 again, then resume from the last checkpoint:

In [ ]:
# Defaults so this cell also works after a Colab restart
DRIVE = '/content/drive/MyDrive/CarVal'
VISION = '/content/CarVal/services/vision'
DATA = '/content/CarDD_yolo/data.yaml'
%cd {VISION}
# from ultralytics import YOLO
# YOLO(f'{VISION}/runs/detect/finetune_yolov8s/weights/last.pt').train(resume=True)

## 5. Evaluate on the held-out test split
Writes precision, recall, mAP@50 and mAP@50-95 (overall and per class) to `reports/`.

In [ ]:
# Defaults so this cell also works after a Colab restart
DRIVE = '/content/drive/MyDrive/CarVal'
VISION = '/content/CarVal/services/vision'
DATA = '/content/CarDD_yolo/data.yaml'
%cd {VISION}
import torch
DEV = 0 if torch.cuda.is_available() else 'cpu'
print('device:', DEV)
import os
if os.path.exists('evaluation/evaluate.py'):
    !python evaluation/evaluate.py --data {DATA} \
        --weights runs/detect/baseline_yolov8n/weights/best.pt:baseline \
        --weights runs/detect/finetune_yolov8s/weights/best.pt:finetune \
        --device auto --out {DRIVE}/reports
else:
    # evaluate.py not in this copy of the code yet: plain Ultralytics test-split metrics
    from ultralytics import YOLO
    for name in ['baseline_yolov8n', 'finetune_yolov8s']:
        print(f'===== {name}')
        YOLO(f'runs/detect/{name}/weights/best.pt').val(data=DATA, split='test', device=DEV, plots=False)

## 6. Download the weights
Download `best.pt` from `MyDrive/CarVal/runs/detect/finetune_yolov8s/weights/` and put it at
`services/vision/weights/best.pt` on your Mac (also copy `weights/model_info.json` from below).

In [ ]:
# Defaults so this cell also works after a Colab restart
DRIVE = '/content/drive/MyDrive/CarVal'
VISION = '/content/CarVal/services/vision'
DATA = '/content/CarDD_yolo/data.yaml'
%cd {VISION}
!cp {VISION}/weights/best.pt {DRIVE}/best.pt && cp {VISION}/weights/model_info.json {DRIVE}/model_info.json
!ls -la {DRIVE}